# Phase 3 — Baseline Covariate Eligibility Audit

Phases 0–2 are frozen. This notebook audits variable eligibility and causal-role possibilities only; it does not construct a DAG, select treatment/time zero/index ECG, estimate effects, fit propensity scores, or finalize an adjustment set.

Evidence insufficient to classify a variable is recorded as unresolved. The notebook relies on the actual three split parquet files and local project documentation.


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path(r'c:/Users/anura/OneDrive/Desktop/capstone2')
DATA_DIR = ROOT / 'master'
SPLITS = ['train', 'validation', 'test']
frames = {split: pd.read_parquet(DATA_DIR / f'{split}.parquet') for split in SPLITS}
split_rows = {split: len(frame) for split, frame in frames.items()}
df = pd.concat(list(frames.values()), ignore_index=True)
print({'split_rows': split_rows, 'combined_rows': len(df), 'columns': len(df.columns)})


## Variable inventory and eligibility

The inventory is generated from every actual master column, not just the expected handwritten list. It includes type, missingness, cardinality, documented provenance, measurement unit, temporal status, possible treatment/outcome effects, outcome information, leakage risk, causal-role possibilities, and eligibility reason.

The audit categories are A–H as specified. A category A is not assigned without clear temporal evidence.


In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from phase3_baseline_covariate_audit import build_results
results = build_results(df, split_rows)
inventory = pd.DataFrame(results['complete_variable_inventory'])
print(results['dataset_summary'])
display(inventory[['variable','group','data_type','missing_count','cardinality_nonmissing','eligibility_classification','temporal_status']])


## Demographics, history, and ECG provenance

Demographic and comorbidity values are candidates only pending temporal and provenance verification. ECG features and `z_ecg` are linked to a `study_id`, but the final index ECG has not been selected. Their relation to treatment and MI can therefore not be assumed to be baseline.


In [ ]:
display(inventory[inventory['group'].isin(['demographic','comorbidity / prior history'])])
print(json.dumps(results['ecg_provenance_assessment'], indent=2))


## Treatment confounding candidates

These are hypotheses about plausible shared causes, not a final confounder list or adjustment set. A variable is not called a confounder solely because it predicts treatment or outcome; measurement before the relevant baseline and a plausible causal relation to both are still required.


In [ ]:
for treatment, candidates in results['preliminary_confounding_candidates'].items():
    print('\n', treatment)
    display(pd.DataFrame(candidates))


## Phase 3 Decision Record

No final treatment, time zero, index ECG, or adjustment set is selected. The eligible output consists of candidates requiring later verification; all timing, leakage, and causal-role limits remain explicit.


In [ ]:
print(json.dumps(results['phase3_decision_record'], indent=2))
print('\nPhase 4 handoff:')
print(json.dumps(results['phase3_decision_record']['phase4_handoff'], indent=2))
